# PYNQ-Z2 频谱分析仪

这是独立的LIVE服务启动入口。先完整解压board ZIP，关闭其他采集程序。界面提供矩形/Hann、峰值、瀑布、停止恢复、截图与整会话导出。硬件在浏览器点击“开始”时配置一次。REPLAY由PC终端入口启动，不修改本Notebook源码。


In [ ]:
from pathlib import Path
from spectrum_integrity import verify_software
BASE = Path.cwd()
identity = verify_software(BASE)
print('Software package PASS:', identity)


## 启动服务

服务在单独的进程中运行，当前Notebook可保持空闲。遇到端口占用时先使用原服务界面，或在原启动位置结束它；保持同一服务即可完成停止/恢复和所有测试。


In [ ]:
import json, os, socket, subprocess, sys, time, uuid
from urllib.request import urlopen

PORT = 8091
probe = socket.socket()
try:
    probe.bind(('127.0.0.1', PORT))
except OSError as error:
    raise RuntimeError('Port8091 is occupied. Use the existing service or stop it with: '
                       'python3 spectrum_service.py stop --port 8091. Do not launch a second LIVE service.') from error
finally:
    probe.close()
log_dir = BASE / 'spectrum_results'
log_dir.mkdir(exist_ok=True)
server_log = log_dir / ('server_' + uuid.uuid4().hex[:12] + '.log')
with server_log.open('xb') as log:
    spectrum_process = subprocess.Popen(
        [sys.executable, str(BASE / 'spectrum_app.py'), '--mode', 'live',
         '--bind', '0.0.0.0', '--port', str(PORT)], cwd=str(BASE),
        stdin=subprocess.DEVNULL, stdout=log, stderr=subprocess.STDOUT,
        start_new_session=(os.name != 'nt'),
        creationflags=subprocess.CREATE_NO_WINDOW if os.name == 'nt' else 0)
for attempt in range(40):
    if spectrum_process.poll() is not None:
        raise RuntimeError(server_log.read_text(errors='replace'))
    try:
        with urlopen('http://127.0.0.1:{}/api/status'.format(PORT), timeout=1) as response:
            server_status = json.load(response)
        assert server_status['mode'] == 'LIVE'
        break
    except OSError:
        time.sleep(0.25)
else:
    raise RuntimeError('Service startup not confirmed; inspect ' + str(server_log))
print('服务PID:', spectrum_process.pid, '日志:', server_log)
print('电脑浏览器打开 http://192.168.2.99:8091 （IP使用你当前Jupyter地址中的板卡IP）')
print('保留本服务，按README_SPECTRUM.md完成短采集、620秒显示、双音/静音，最后一次性导出会话ZIP。')


## 全部测试结束后：结束服务

先在浏览器停止并导出结果，再运行此格。若出现无法安全停止或缓冲区保留提示，保留服务和日志，按提示重启板卡；不要强制释放缓冲区。


In [ ]:
import signal
if spectrum_process.poll() is None:
    spectrum_process.send_signal(signal.SIGINT)
    try:
        spectrum_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        raise RuntimeError('Service remains alive: inspect its log and any unsafe-stop warning before rebooting the board')
print('服务结束，返回码:', spectrum_process.returncode)
